# 03 - Build the day-28 feature matrix

Joins labels, demographics, engagement, forum proxy, and early assessment features. Saves a Parquet file for Week 11 modelling.

In [16]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Open from notebooks/ or project root.")
sys.path.insert(0, str(PROJECT_ROOT))

In [17]:
# ## Diagnostic — verify which features.py is loaded
#
# If this cell does not show the fixed code (i.e. `vle[["id_site", "activity_type"]]`
# inside `engagement_features`), the problem is a stale file or a stale `.pyc`

In [18]:
import importlib
import inspect

import src.features as features_mod
importlib.reload(features_mod)

print("Loaded from:", features_mod.__file__)
print()
print("--- engagement_features source (first 60 lines) ---")
src_lines = inspect.getsource(features_mod.engagement_features).splitlines()
for i, line in enumerate(src_lines[:60], start=1):
    print(f"{i:3d}  {line}")

Loaded from: /home/pa11ie/Projects/cosc329/atRiskPrediction/src/features.py

--- engagement_features source (first 60 lines) ---
  1  def engagement_features(
  2      student_vle_trunc: pd.DataFrame,
  3      vle: pd.DataFrame,
  4      cutoff_day: int = CUTOFF_DAY,
  5  ) -> pd.DataFrame:
  6      """Overall engagement plus per-activity-type click totals.
  7  
  8      ``studentVle`` already carries ``code_module`` and ``code_presentation``,
  9      and ``id_site`` uniquely maps to (module, presentation, activity_type)
 10      in ``vle``. We therefore pull only ``id_site`` and ``activity_type``
 11      to avoid merge-suffix collisions on the KEY columns.
 12      """
 13      missing = [c for c in KEY if c not in student_vle_trunc.columns]
 14      if missing:
 15          raise KeyError(
 16              f"student_vle_trunc is missing required columns {missing}. "
 17              f"Present columns: {list(student_vle_trunc.columns)}"
 18          )
 19  
 20      sv = student_vl

In [19]:
# Confirm the columns that survive truncation
import pandas as pd
from src.data_loader import load_all
from src.cutoff import CUTOFF_DAY, truncate_student_vle

tables = load_all()
sv_trunc = truncate_student_vle(tables["studentVle"])
print("CUTOFF_DAY:", CUTOFF_DAY)
print("student_vle_trunc columns:", list(sv_trunc.columns))
assert "code_module" in sv_trunc.columns, "code_module missing from studentVle!"
assert "code_presentation" in sv_trunc.columns, "code_presentation missing from studentVle!"

CUTOFF_DAY: 28
student_vle_trunc columns: ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']


In [20]:
# Build the full feature matrix
from src.features import build_feature_matrix

features = build_feature_matrix(
    student_info=tables["studentInfo"],
    student_vle=tables["studentVle"],
    vle=tables["vle"],
    student_assessment=tables["studentAssessment"],
    assessments=tables["assessments"],
)

print(features.shape)
display(features.head())


(32593, 45)


,id_student,code_module,code_presentation,final_result,at_risk,gender,region,highest_education,imd_band,age_band,...,clicks_url,forum_clicks,forum_active_days,early_submitted,early_mean_score,early_min_score,early_max_score,early_due,early_submission_rate,early_missing
0,11391,AAA,2013J,Pass,0,M,East Anglian Region,HE Qualification,90-100%,55<=,...,1.0,55.0,4.0,1.0,78.0,78.0,78.0,1.0,1.0,0.0
1,28400,AAA,2013J,Pass,0,F,Scotland,HE Qualification,20-30%,35-55,...,30.0,209.0,14.0,1.0,70.0,70.0,70.0,1.0,1.0,0.0
2,30268,AAA,2013J,Withdrawn,1,F,North Western Region,A Level or Equivalent,30-40%,35-55,...,4.0,126.0,9.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
3,31604,AAA,2013J,Pass,0,F,South East Region,A Level or Equivalent,50-60%,35-55,...,17.0,131.0,12.0,1.0,72.0,72.0,72.0,1.0,1.0,0.0
4,32885,AAA,2013J,Pass,0,F,West Midlands Region,Lower Than A Level,50-60%,0-35,...,6.0,138.0,17.0,1.0,69.0,69.0,69.0,1.0,1.0,0.0


In [21]:
# Class balance — motivates SMOTE in Week 11
print(features["at_risk"].value_counts(dropna=False))
print(features["at_risk"].value_counts(normalize=True).round(3))

at_risk
1    17208
0    15385
Name: count, dtype: int64
at_risk
1    0.528
0    0.472
Name: proportion, dtype: float64


In [22]:
# Column-level NaN rates that will need handling in Week 11
na = features.isna().mean().sort_values(ascending=False)
display(na[na > 0].round(3))

early_max_score     0.372
early_min_score     0.372
early_mean_score    0.372
early_due           0.371
imd_band            0.034
dtype: float64

In [24]:
# Save as CSV (no extra dependency required)
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "features_day28.csv"
features.to_csv(out_path, index=False)
print(f"Saved {features.shape[0]:,} rows × {features.shape[1]} columns to {out_path}")

Saved 32,593 rows × 45 columns to /home/pa11ie/Projects/cosc329/atRiskPrediction/data/processed/features_day28.csv
